In [33]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [34]:
df = pd.read_csv(
"/content/fashion-mnist_train.csv",
    engine="python",
    on_bad_lines="skip"
)

# remove broken rows
df = df.dropna()

# force numeric conversion (removes hidden corruption)
df = df.apply(pd.to_numeric, errors="coerce")
df = df.dropna()

# keep only valid FashionMNIST labels (0–9)
df = df[df["label"].between(0, 9)]

print("Final shape:", df.shape)
print("Labels:", np.unique(df["label"]))

Final shape: (33115, 785)
Labels: [0 1 2 3 4 5 6 7 8 9]


In [35]:
X = df.iloc[:, 1:].values.astype(np.float32)
y = df.iloc[:, 0].values.astype(np.int64)

# normalize pixel values (0–255 → 0–1)
X = X / 255.0

In [36]:
class FMNISTDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = FMNISTDataset(X, y)

In [37]:
train_loader = DataLoader(dataset, batch_size=64, shuffle=True,pin_memory_device=True)

In [38]:
class FMNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)   # 10 classes (VERY IMPORTANT)
        )

    def forward(self, x):
        return self.net(x)

model = FMNISTModel()
model=model.to(device)

In [39]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [40]:
epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # forward
        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

Epoch 1/100, Loss: 310.9412
Epoch 2/100, Loss: 211.8671
Epoch 3/100, Loss: 190.6469
Epoch 4/100, Loss: 173.1762
Epoch 5/100, Loss: 163.7828
Epoch 6/100, Loss: 154.6402
Epoch 7/100, Loss: 147.4830
Epoch 8/100, Loss: 140.5518
Epoch 9/100, Loss: 132.0841
Epoch 10/100, Loss: 127.7806
Epoch 11/100, Loss: 124.1491
Epoch 12/100, Loss: 118.3304
Epoch 13/100, Loss: 110.1129
Epoch 14/100, Loss: 109.7810
Epoch 15/100, Loss: 102.3315
Epoch 16/100, Loss: 99.2444
Epoch 17/100, Loss: 96.1016
Epoch 18/100, Loss: 90.6416
Epoch 19/100, Loss: 86.4171
Epoch 20/100, Loss: 86.9360
Epoch 21/100, Loss: 80.4382
Epoch 22/100, Loss: 77.8852
Epoch 23/100, Loss: 76.3403
Epoch 24/100, Loss: 73.2861
Epoch 25/100, Loss: 72.0910
Epoch 26/100, Loss: 68.0483
Epoch 27/100, Loss: 66.0619
Epoch 28/100, Loss: 63.1290
Epoch 29/100, Loss: 64.4581
Epoch 30/100, Loss: 59.7903
Epoch 31/100, Loss: 55.6176
Epoch 32/100, Loss: 55.9207
Epoch 33/100, Loss: 55.4399
Epoch 34/100, Loss: 54.9375
Epoch 35/100, Loss: 50.1520
Epoch 36/100, 

In [41]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)

        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

accuracy = 100 * correct / total

print(f"Training Accuracy: {accuracy:.2f}%")

Training Accuracy: 98.65%
